In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Raw/loan.csv")




C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\811700591.py:4: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Raw/loan.csv")


In [3]:
df.shape
len(df.columns)


111

In [4]:
df.drop(columns=['id', 'member_id'], inplace=True)


In [5]:
leakage_cols = [
    'out_prncp','out_prncp_inv','total_pymnt','total_pymnt_inv',
    'total_rec_prncp','total_rec_int','total_rec_late_fee',
    'recoveries','collection_recovery_fee','last_pymnt_amnt'
]

df.drop(columns=leakage_cols, inplace=True, errors='ignore')


In [6]:
df['loan_status'].value_counts()


loan_status
Fully Paid     32950
Charged Off     5627
Current         1140
Name: count, dtype: int64

In [7]:
df = df[df['loan_status'].isin(['Fully Paid','Charged Off'])]

df['loan_status'] = df['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})


In [8]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
missing_pct.head(20)


num_rev_tl_bal_gt_0      1.0
num_rev_accts            1.0
pct_tl_nvr_dlq           1.0
mo_sin_old_il_acct       1.0
mo_sin_old_rev_tl_op     1.0
mo_sin_rcnt_rev_tl_op    1.0
mo_sin_rcnt_tl           1.0
mths_since_recent_bc     1.0
mort_acc                 1.0
bc_open_to_buy           1.0
bc_util                  1.0
total_rev_hi_lim         1.0
all_util                 1.0
max_bal_bc               1.0
avg_cur_bal              1.0
acc_open_past_24mths     1.0
inq_last_12m             1.0
total_cu_tl              1.0
inq_fi                   1.0
mths_since_rcnt_il       1.0
dtype: float64

In [9]:
high_missing = missing_pct[missing_pct > 0.7].index
df.drop(columns=high_missing, inplace=True)


In [10]:
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


In [11]:
emp_cols = [c for c in df.columns if c.startswith('emp_title_')]
len(emp_cols)


0

In [12]:
df.drop(columns=emp_cols, inplace=True)


In [14]:
'total_rev_hi_lim' in df.columns


False

In [16]:
df.replace([np.inf, -np.inf], 0, inplace=True)


In [17]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


ValueError: could not convert string to float: ' 60 months'

In [20]:
X_train.dtypes[X_train.dtypes == 'object']


term                   object
int_rate               object
grade                  object
sub_grade              object
emp_title              object
emp_length             object
home_ownership         object
verification_status    object
issue_d                object
pymnt_plan             object
url                    object
desc                   object
purpose                object
title                  object
zip_code               object
addr_state             object
earliest_cr_line       object
revol_util             object
initial_list_status    object
last_pymnt_d           object
last_credit_pull_d     object
application_type       object
dtype: object

In [21]:
    df['term'] = df['term'].str.extract('(\d+)').astype(int)


In [22]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [23]:
X_train.select_dtypes(include='object').columns


Index(['int_rate', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'issue_d', 'pymnt_plan', 'url',
       'desc', 'purpose', 'title', 'zip_code', 'addr_state',
       'earliest_cr_line', 'revol_util', 'initial_list_status', 'last_pymnt_d',
       'last_credit_pull_d', 'application_type'],
      dtype='object')

In [25]:
df['int_rate'] = (
    df['int_rate']
    .str.replace('%', '', regex=False)
    .astype(float)
)


In [26]:
df['revol_util'] = (
    df['revol_util']
    .str.replace('%', '', regex=False)
    .astype(float)
)


In [27]:
df['emp_length'] = (
    df['emp_length']
    .str.replace('years', '', regex=False)
    .str.replace('year', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.replace('<', '', regex=False)
    .str.strip()
)

df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')
df['emp_length'].fillna(df['emp_length'].median(), inplace=True)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\605947800.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['emp_length'].fillna(df['emp_length'].median(), inplace=True)


In [28]:
drop_text_cols = [
    'emp_title', 'url', 'desc', 'title', 'zip_code'
]

df.drop(columns=drop_text_cols, inplace=True, errors='ignore')


In [29]:
cat_cols = [
    'grade', 'sub_grade', 'home_ownership', 'verification_status',
    'purpose', 'addr_state', 'initial_list_status',
    'application_type'
]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [30]:
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['credit_history_length'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 365
)

df.drop(columns=date_cols, inplace=True)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\113381075.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\113381075.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\113381075.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Rudra\AppData\Local\Temp\ipykernel_7372\113381075.py:4: UserWarning: Could not infer 

In [31]:
df.drop(columns=['pymnt_plan'], inplace=True, errors='ignore')


In [32]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

X.select_dtypes(include='object').columns


Index([], dtype='object')

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


C:\Users\Rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\Rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\Rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [34]:
df.isnull().sum().sum()


np.int64(38627)

In [35]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing


credit_history_length    38577
revol_util                  50
dtype: int64

In [39]:
df['revol_util'] = df['revol_util'].fillna(0)



In [40]:
df['credit_history_length'] = df['credit_history_length'].fillna(
    df['credit_history_length'].median()
)



C:\Users\Rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [41]:
df.isnull().sum().sum()


np.int64(38577)

In [42]:
empty_cols = df.columns[df.isnull().all()]
empty_cols


Index(['credit_history_length'], dtype='object')

In [43]:
df.drop(columns=empty_cols, inplace=True)


In [44]:
df = df.fillna(df.median())


In [45]:
df.isnull().sum().sum()


np.int64(0)

In [46]:
df['loan_status'].value_counts(normalize=True)


loan_status
0    0.854136
1    0.145864
Name: proportion, dtype: float64

In [47]:
leakage_check = [
    'total_pymnt','total_rec_prncp','recoveries',
    'collection_recovery_fee','last_pymnt_amnt',
    'out_prncp'
]

set(leakage_check).intersection(df.columns)


set()

In [48]:
X = df.drop('loan_status', axis=1)
X.dtypes.unique()


array([dtype('int64'), dtype('float64'), dtype('bool')], dtype=object)

In [49]:
df.shape


(38577, 135)

In [50]:
df.to_csv("../data/Processed/processed_loan_data.csv", index=False)


In [51]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

X.to_csv("../data/Processed/X_processed.csv", index=False)
y.to_csv("../data/Processed/y_target.csv", index=False)


In [52]:
import joblib
joblib.dump(scaler, "../data/Processed/standard_scaler.pkl")


['../data/Processed/standard_scaler.pkl']